In [ ]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime, timedelta
import ta
from functools import reduce
import plotly.io as pio

In [ ]:
from utils.generic_utils import SQLModule
from utils.data_utils import StockPriceProcess
from utils.constants import CURRENCY_MAPPER, PLOTLY_CURRENCY_NORMALIZER

In [ ]:
CODES = {
    'vietnam' : ['ACB'],
    'australia' : ['TPG', 'TNE', 'SGLLV']
}
CODES_TO_COUNTRY = {v : k for k,vv in CODES.items() for v in vv}
FEATURES = ['high','low','close', 'volume', 'acc_dist']
DAYS = 365

In [ ]:
# Get 1 year data
end_date = datetime.today().date()
start_date = end_date - timedelta(days = DAYS)
world_engine = SQLModule.get_engine(country = 'world')
# Get all exchange rate
query = f"""
    SELECT
        *
    FROM daily_average_exchange_rate_usd_based
    WHERE
        date >= DATE '{start_date}'
        AND
        date <= DATE '{end_date}'
    ORDER BY date
"""
ex_rate = pd.read_sql_query(query, world_engine)
ex_rate.set_index('date', inplace = True)
# fill nan for each rate
for col in ex_rate.columns:
    # Backfilling the variables
    ex_rate[col] = ex_rate[col].fillna(method = 'bfill').fillna(method = 'ffill')

In [ ]:
dfs = []
for stock_code,country in CODES_TO_COUNTRY.items():
    engine = SQLModule.get_engine(country = country)
    stock_query = f"""
        SELECT
            date,
            high,
            low,
            close,
            volume
        FROM transaction
        WHERE
            stock_code = '{stock_code}'
            AND
            date >= DATE '{start_date}'
            AND
            date <= DATE '{end_date}'
        ORDER BY date
    """
    df = pd.read_sql_query(stock_query, engine)
    df.set_index('date', inplace = True)
    # convert to usd
    if country != 'united_states':
        df = df.join(ex_rate[[CURRENCY_MAPPER[country]]])
        for price_type in ['high', 'low', 'close']:
            df[price_type] = df[price_type] / df[CURRENCY_MAPPER[country]]
    df = StockPriceProcess.remove_invalid_data(df, country = country)
    acc_dist_obj = ta.volume.AccDistIndexIndicator(high = df['high'], low = df['low'], close = df['close'], volume = df['volume'])

    df['acc_dist'] = acc_dist_obj.acc_dist_index()

    df = df[FEATURES]

    # Change column to multi-index
    df.columns = pd.MultiIndex.from_tuples([(stock_code,col) for col in df.columns])

    dfs.append(df.reset_index())
result = reduce(lambda l,r: pd.merge(l,r, on='date', how='outer'), dfs).sort_values(by='date').reset_index(drop = True)
result.index = result['date']
df = result.drop('date', axis = 1)
df.iloc[-5:,:]

ACB                                                     TPG  \
                high       low     close      volume      acc_dist      high   
date                                                                           
2025-02-17  1.022862  1.014978  1.016949   5754831.0  1.370441e+08  2.803221   
2025-02-18  1.026451  1.016581  1.016581   5307415.0  1.317367e+08  2.818586   
2025-02-19  1.026451  1.016581  1.016581   5423705.0  1.263130e+08  2.818586   
2025-02-20  1.026451  1.016581  1.016581   6736703.0  1.195763e+08  2.818586   
2025-02-21  1.026451  1.016581  1.016581  12719310.0  1.068570e+08  2.818586   

                                                               TNE             \
                 low     close     volume      acc_dist       high        low   
date                                                                            
2025-02-17  2.758726  2.784152   724608.0 -4.951698e+06  20.747653  20.302698   
2025-02-18  2.786774  2.812224   619835.0 -4.579792e+06  20.569954  20.290002   
2025-02-19  2.786774  2.812224   832449.0 -4.080316e+06  20.569954  20.290002   
2025-02-20  2.786774  2.812224   959568.0 -3.504568e+06  20.569954  20.290002   
2025-02-21  2.786774  2.812224  1481162.0 -2.615859e+06  20.569954  20.290002   

                                                   SGLLV                      \
                close     volume      acc_dist      high       low     close   
date                                                                           
2025-02-17  20.366263   603258.0  3.095559e+07  6.706120  6.470929  6.706120   
2025-02-18  20.487241   488013.0  3.115523e+07  6.756972  6.648809  6.680622   
2025-02-19  20.487241  1214257.0  3.165197e+07  6.756972  6.648809  6.680622   
2025-02-20  20.487241   600200.0  3.189751e+07  6.756972  6.648809  6.680622   
2025-02-21  20.487241   908811.0  3.226929e+07  6.756972  6.648809  6.680622   

                                    
             volume       acc_dist  
date                                
2025-02-17  20391.0 -469202.984858  
2025-02-18  14174.0 -475039.309735  
2025-02-19  12201.0 -480063.226754  
2025-02-20  19045.0 -487905.247870  
2025-02-21  10906.0 -492395.932159

In [ ]:
for num,(stock_code,country) in enumerate(CODES_TO_COUNTRY.items()):
    fig = make_subplots(rows = 2, cols = 1, specs=[[{}],[{"secondary_y": True}]])
    # Get data from that stock code
    _df = df[[(stock_code, col) for col in FEATURES]].droplevel(0, axis = 1)
    # merge with ex_rate
    _df = _df.join(ex_rate[[CURRENCY_MAPPER[country]]])
    for col in ['high', 'low', 'close']:
        _df[col] = _df[col] * _df[CURRENCY_MAPPER[country]]
    # Plot the price
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['close'] / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  
            marker = dict(color = 'blue'),
            name = 'Close price',
            showlegend = False
        ),
        row = 1, col = 1
    )
    fig.update_yaxes(
        title = 'Close price',
        tickprefix = f'{CURRENCY_MAPPER[CODES_TO_COUNTRY[stock_code]]} ',
        ticksuffix = f' * {PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]]}', 
        showgrid = True, 
        gridcolor = 'gray',
        row = 1, col = 1
    )

    # Plot the volume
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['volume'],  
            marker = dict(color = 'black'),
            name = 'Volume',
            showlegend = False
        ),
        row = 2, col = 1, secondary_y = False
    )
    fig.update_yaxes(
        title = 'Volume', 
        showgrid = True, 
        gridcolor = 'gray', 
        tickfont=dict(color='black'), 
        titlefont=dict(color='black'), 
        row = 2, col = 1, secondary_y = False
    )

    # Plot the A/D index
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['acc_dist'],  
            marker = dict(color = 'purple'),
            name = 'A/D index',
            showlegend = False
        ),
        row = 2, col = 1, secondary_y = True
    )
    fig.update_yaxes(
        title = 'A/D index', 
        tickmode="sync", 
        tickfont=dict(color='purple'), 
        titlefont=dict(color='purple'), 
        row = 2, col = 1, 
        secondary_y = True
    )
    fig.update_xaxes(title = 'Date', showgrid = False)
    fig.update_layout(
        plot_bgcolor = 'white' if num % 2 == 0 else "rgb(210, 210, 210)",
        paper_bgcolor = 'white' if num % 2 == 0 else "rgb(210, 210, 210)",
        height = 800,
        font = dict(size = 20),
        title = f'[{country}] {stock_code}'
    )
    fig.show()